# Test for harmful factors

## Coefficient check

In [12]:
from src.model import load_model
import pandas as pd

model = load_model("models/baseline.joblib")

classifier = model.named_steps['classifier']

weights = pd.Series(
    classifier.coef_[0], 
    index=['gold_diff', 'xp_diff', 'cs_diff', 'objective_diff', 'first_blood', 'herald_diff']
)

print("Model Feature Weights:")
print(weights.sort_values(ascending=False))

Model Feature Weights:
gold_diff         0.995972
xp_diff           0.488475
objective_diff    0.292757
first_blood       0.045199
cs_diff          -0.088472
herald_diff      -0.120032
dtype: float64


## Variance Inflation Factor (VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from src.features import engineer_features

df = pd.read_csv('data/raw/high_diamond_ranked_10min.csv')
processed_df = engineer_features(df)

X_continuous = processed_df[['gold_diff', 'xp_diff', 'kill_diff', 'cs_diff']]

vif_data = pd.DataFrame()
vif_data["Feature"] = X_continuous.columns
vif_data["VIF"] = [variance_inflation_factor(X_continuous.values, i) for i in range(len(X_continuous.columns))]

print(vif_data.sort_values(by="VIF", ascending=False))

     Feature       VIF
0  gold_diff  5.285694
1    xp_diff  5.140673
2    cs_diff  1.732079


'gold_diff' has a massive VIF which is causing problems with 'kill_diff'.

'kill_diff' should have a positive coefficent as kills will increase the chance of winning but some of the impact from kills is being taken as 'gold_diff'.

In [9]:
X_continuous = processed_df[['gold_diff', 'xp_diff', 'cs_diff']]

vif_data = pd.DataFrame()
vif_data["Feature"] = X_continuous.columns
vif_data["VIF"] = [variance_inflation_factor(X_continuous.values, i) for i in range(len(X_continuous.columns))]

print(vif_data.sort_values(by="VIF", ascending=False))

     Feature       VIF
0  gold_diff  5.285694
1    xp_diff  5.140673
2    cs_diff  1.732079


When 'kill_diff' is removed the factors have much more normal VIF so we must remove 'kill_diff'